# SEC 8-K New Product Extraction with a Local LLM
**CAP5619 — LLM-based Named Entity Recognition on SEC Form 8-K filings**  
**Author:** Frank Soto

This notebook extracts **new product releases** from SEC 8-K filings of S&P 500 companies and saves them as a CSV with the columns
`company_name, stock_name, filing_time, new_product, product_description`.

| Component | Choice |
|---|---|
| Data source | SEC EDGAR: `company_tickers.json` (ticker → CIK) and the browse-edgar Atom feed (CIK → 8-Ks) |
| Parsing | `requests` + `BeautifulSoup` (XML feed, filing index, EX-99 press-release exhibits) |
| NER (candidates) | spaCy `en_core_web_sm` (ORG / PRODUCT hints) |
| LLM (extraction + final NER) | **Llama 3.1 8B** running locally through **Ollama**, JSON-schema constrained output |

All logic lives in `sec_8k_product_extractor.py`. This notebook imports it, demonstrates each step on one filing, and then runs the full pipeline. HTTP responses and LLM answers are cached in `cache/`, so re-running the notebook is fast.

## 0. Setup
Requires `ollama serve` running and `ollama pull llama3.1:8b`. `SEC_USER_AGENT` is set to the author's name and e-mail (SEC fair-access policy).

In [1]:
import os
# SEC fair-access policy: identify yourself (must be set before importing the extractor)
os.environ["SEC_USER_AGENT"] = "Frank Soto fsoto@ucf.edu"
import json, requests
import pandas as pd
import sec_8k_product_extractor as ex

print("SEC User-Agent:", ex.USER_AGENT)
print("Model:", ex.MODEL)
print("Ollama models:", [m["name"] for m in requests.get("http://localhost:11434/api/tags").json()["models"]])
print("spaCy NER loaded:", ex.NLP is not None)

SEC User-Agent: Frank Soto fsoto@ucf.edu
Model: llama3.1:8b
Ollama models: ['llama3.1:8b']
spaCy NER loaded: True


## 1. Ticker → CIK
Uses `https://www.sec.gov/files/company_tickers.json`.

In [2]:
ticker_map = ex.load_ticker_map()
print(len(ticker_map), "tickers in the SEC map")
ticker_map["AAPL"]

10409 tickers in the SEC map


{'cik': '0000320193', 'title': 'Apple Inc.'}

## 2. CIK → recent 8-K filings
Uses `https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={cik}&type=8-K&count={count}&output=atom`, parsed with BeautifulSoup's XML parser. The `<updated>` element gives the acceptance timestamp used as **Filing Time**.

In [3]:
filings = ex.get_8k_filings(ticker_map["AAPL"]["cik"], count=10)
pd.DataFrame(filings)[["accession", "filing_date", "updated", "items"]]

,accession,filing_date,updated,items
0,0000320193-26-000024,2026-11-02,2026-11-02T16:30:23-05:00,items 2.02 and 9.01
1,0000320193-26-000018,2026-07-30,2026-07-30T16:30:28-04:00,items 2.02 and 9.01
2,0000320193-26-000011,2026-04-30,2026-04-30T16:30:41-04:00,items 2.02 and 9.01
3,0001140361-26-015711,2026-04-20,2026-04-20T17:29:51-04:00,item 5.02
4,0001140361-26-006577,2026-02-24,2026-02-24T16:55:58-05:00,items 5.07 and 9.01
5,0000320193-26-000005,2026-01-29,2026-01-29T16:30:33-05:00,items 2.02 and 9.01
6,0001140361-26-000199,2026-01-02,2026-01-02T16:30:52-05:00,item 5.02
7,0001140361-25-044561,2025-12-05,2025-12-05T16:31:42-05:00,item 5.02
8,0000320193-25-000077,2025-10-30,2025-10-30T16:30:35-04:00,items 2.02 and 9.01


## 3. Download the press release (Exhibit 99) and preprocess
Only 8-Ks with items 2.02, 7.01 or 8.01 are kept, as they have the press releases. Financial tables are dropped, and only the sentences around launch keywords (*launched, introduced, unveiled, now available, FDA approved…*) go to the LLM. This cuts prompt size by about 80–90%.

In [4]:
filing = next(f for f in filings if any(i in f["items"] for i in ex.PRESS_RELEASE_ITEMS))
text = ex.get_press_release_text(filing)
passage = ex.extract_product_passages(text)
print(f"Full exhibit text: {len(text):,} chars  ->  product passages: {len(passage):,} chars\n")
print(passage[:1500], "...")

Full exhibit text: 6,161 chars  ->  product passages: 881 chars

EX-99.1 2 a8-kex991q4202609262026.htm EX-99.1 Document Exhibit 99.1 Apple reports fourth quarter results September quarter records for total company revenue, iPhone revenue, and EPS Services revenue reaches new all-time high CUPERTINO, CALIFORNIA — Apple ® today announced financial results for its fiscal 2026 fourth quarter ended September 26, 2026. The Company posted quarterly revenue of $114.5 billion, up 12 percent year over year. That includes the iPhone Duo, the most transformational iPhone since the original, alongside iPhone 18 Pro and iPhone 18 Pro Max, our most powerful iPhones yet. We also just introduced a whole new generation of hardware for the home and brought Touch to Mac for the first time. Together with the all-new Siri AI, it’s an incredible lineup to carry us into the holiday season.” “Apple delivered September quarter records for net income and EPS. ...


## 4. Named Entity Recognition
**Stage 1, spaCy:** a statistical NER model proposes ORG and PRODUCT candidates. 

**Stage 2, LLM:** Llama 3.1 does the final entity labeling. It is prompted as an NER system with two entity types (ORGANIZATION = filer, PRODUCT = newly launched product) and must answer in a fixed JSON schema.

In [5]:
print(ex.spacy_entities("Apple Inc. has announced the launch of its latest MacBook Pro, featuring a new M3 processor."))
ents = ex.spacy_entities(passage)
ents

{'ORG': ['Apple Inc.'], 'PRODUCT': ['MacBook Pro']}


{'ORG': ['Apple', 'Company', 'EPS', 'EPS Services', 'Siri AI', 'iPhones'],
 'PRODUCT': ['Touch to Mac']}

In [6]:
print(ex.SYSTEM_PROMPT)

You are a financial-document Named Entity Recognition (NER) system.
You read excerpts of SEC Form 8-K press releases and extract NEW PRODUCT releases.

Entity types:
- ORGANIZATION: the company that filed the 8-K.
- PRODUCT: a specific, named product, service, drug, device, model, platform or
  software that the company says it launched, introduced, unveiled, released,
  began shipping, made available, or received approval to sell, in this document.

Rules:
- Only include products that are NEW (launched/announced/approved recently).
- Do NOT include: financial results, revenue lines, business segments, generic
  categories ("AI solutions"), acquisitions, executives, dividends, buybacks,
  partnerships without a named product, or products merely mentioned in passing.
- Facilities, campuses, data centers, contracts and customer deals are NOT products.
- Use the product name exactly as written in the text.
- company_name: the company's full name as written in the press release
  (e.g. "Ap

In [7]:
result = ex.query_llm("AAPL", passage, ents)
print(json.dumps(result, indent=2))

{
  "company_name": "Apple Inc.",
  "products": [
    {
      "product_name": "iPhone Duo",
      "product_description": "The most transformational iPhone since the original, with new features and capabilities.",
      "launch_evidence": "That includes the iPhone Duo, the most transformational iPhone since the original, alongside iPhone 18 Pro and iPhone 18 Pro Max, our most powerful iPhones yet."
    },
    {
      "product_name": "iPhone 18 Pro",
      "product_description": "Our most powerful iPhone yet, with advanced features and capabilities.",
      "launch_evidence": "That includes the iPhone Duo, the most transformational iPhone since the original, alongside iPhone 18 Pro and iPhone 18 Pro Max, our most powerful iPhones yet."
    },
    {
      "product_name": "iPhone 18 Pro Max",
      "product_description": "Our most powerful iPhone yet, with advanced features and capabilities.",
      "launch_evidence": "That includes the iPhone Duo, the most transformational iPhone since th

## 5. Validation rules
* The product name appears verbatim in filing text. This rejects hallucinated names.
* `launch_evidence` must contain a launch keyword or the word *new*. 
* Generic names are rejected: exact matches such as "platform" or "solutions", and category phrases ending in solutions / tools / skills / offerings / capabilities.
* Fragment descriptions (< 25 chars) are rejected.
* Descriptions are cut at a word boundary to **< 180 characters**.
* At most 5 products per document, de-duplicated per company.
* Company names cleaned up: each ticker gets the company name the LLM extracted most often, all-caps SEC names are cleaned up (e.g., "ADVANCED MICRO DEVICES INC" → "Advanced Micro Devices, Inc.").

In [8]:
for p in result["products"]:
    print(p["product_name"], "| in text:", ex.product_in_text(p["product_name"], text),
          "| desc len:", len(ex.truncate(p["product_description"])))

iPhone Duo | in text: True | desc len: 88
iPhone 18 Pro | in text: True | desc len: 70
iPhone 18 Pro Max | in text: True | desc len: 70
Touch to Mac | in text: True | desc len: 58
Siri AI | in text: True | desc len: 45


## 6. Full run (S&P 500 companies, stop at ≥ 100 products)
If the SEC returns repeated errors or timeouts for a company or filing (Texas Instruments returned HTTP 503 during development), the pipeline logs a `[skip]` line and moves on instead of stopping. Cached results make this fast. Delete `cache/llm_cache.json` to re-query the LLM from scratch (about 20 s per filing on an Apple M4 Max).

In [9]:
rows, stats = ex.run(target_rows=110, filings_per_company=20)


== AAPL (Apple Inc.) - 19 8-K filings
   2026-11-02 0000320193-26-000024: 2 product(s)  [total 2]
   2026-07-30 0000320193-26-000018: 0 product(s)  [total 2]
   2026-04-30 0000320193-26-000011: 2 product(s)  [total 4]
   2025-10-30 0000320193-25-000077: 4 product(s)  [total 8]


   2025-07-31 0000320193-25-000071: 0 product(s)  [total 8]


   2025-05-01 0000320193-25-000055: 1 product(s)  [total 9]



== MSFT (MICROSOFT CORP) - 20 8-K filings


   2026-10-28 0001193125-26-436149: 0 product(s)  [total 9]
   2026-09-02 0001193125-26-380280: 0 product(s)  [total 9]


   2024-12-03 0000950170-24-132722: 2 product(s)  [total 11]

== NVDA (NVIDIA CORP) - 20 8-K filings
   2026-08-26 0001045810-26-000073: 2 product(s)  [total 13]
   2026-08-17 0001045810-26-000069: 0 product(s)  [total 13]
   2026-05-20 0001045810-26-000051: 1 product(s)  [total 14]


... (21 outputs left out)

   2025-05-14 0001193125-25-119864: 0 product(s)  [total 93]
   2025-02-12 0001193125-25-025114: 0 product(s)  [total 93]

== IBM (INTERNATIONAL BUSINESS MACHINES CORP) - 20 8-K filings
   2026-10-21 0000051143-26-000093: 0 product(s)  [total 93]


   2026-07-22 0000051143-26-000077: 0 product(s)  [total 93]
   2026-07-14 0000051143-26-000070: 2 product(s)  [total 95]
   2026-01-28 0000051143-26-000004: 0 product(s)  [total 95]
   2025-10-23 0000051143-25-000063: 1 product(s)  [total 96]
   2025-10-22 0000051143-25-000061: 0 product(s)  [total 96]


   2025-07-24 0000051143-25-000051: 1 product(s)  [total 97]
   2025-07-23 0000051143-25-000048: 0 product(s)  [total 97]

== NOW (ServiceNow, Inc.) - 19 8-K filings
   2026-10-28 0001373715-26-000092: 4 product(s)  [total 101]
   2026-07-22 0001373715-26-000072: 5 product(s)  [total 106]


   2026-04-22 0001373715-26-000054: 2 product(s)  [total 108]
   2026-01-28 0001373715-26-000005: 2 product(s)  [total 110]

Wrote 110 rows to product_releases.csv


In [10]:
df = pd.read_csv(ex.OUTPUT_CSV)
print(df.shape)
print("Max description length:", df.product_description.str.len().max())
pd.set_option("display.max_colwidth", 90)
df.head(25)

(110, 5)
Max description length: 179


,company_name,stock_name,filing_time,new_product,product_description
0,Apple Inc.,AAPL,2026-11-02 16:30:23,Touch to Mac,A new feature that brings Touch to Mac for the first time.
1,Apple Inc.,AAPL,2026-11-02 16:30:23,Siri AI,A new AI-powered assistant for Apple devices.
2,Apple Inc.,AAPL,2026-04-30 16:30:41,M4-powered iPad Air,"The latest iPad Air model, powered by the M4 chip."
3,Apple Inc.,AAPL,2026-04-30 16:30:41,MacBook Neo,"A new MacBook model, captivating customers worldwide."
4,Apple Inc.,AAPL,2025-10-30 16:30:35,iPhone 17,"Apple's latest smartphone, part of the best iPhone lineup ever."
5,Apple Inc.,AAPL,2025-10-30 16:30:35,iPhone 17 Pro,"Apple's high-end smartphone, part of the best iPhone lineup ever."
6,Apple Inc.,AAPL,2025-10-30 16:30:35,iPhone Air,"Apple's latest smartphone, part of the best iPhone lineup ever."
7,Apple Inc.,AAPL,2025-10-30 16:30:35,AirPods Pro 3,"Apple's latest wireless earbuds, featuring improved sound and design."
8,Apple Inc.,AAPL,2025-05-01 16:30:21,new Macs,"Apple's latest line of Mac computers, powered by Apple silicon."
9,Microsoft,MSFT,2024-12-03 16:00:37,Azure Maia,"Custom-built cloud infrastructure for AI workloads, including a custom AI accelerator."


## 7. Output
* `product_releases.csv`: required CSV, all values quoted
* `product_releases_table.txt`: pipe-separated table in the assignment's display format
* `run_stats.json`: counts used in the report

In [11]:
print(open(ex.OUTPUT_TABLE).read()[:3000])

Company Name                 | Stock Name | Filing Time         | New Product                      | Product Description
--------------------------------------------------------------------------------------------------------------------------------------------
Apple Inc.                   | AAPL       | 2026-11-02 16:30:23 | Touch to Mac                     | A new feature that brings Touch to Mac for the first time.
Apple Inc.                   | AAPL       | 2026-11-02 16:30:23 | Siri AI                          | A new AI-powered assistant for Apple devices.
Apple Inc.                   | AAPL       | 2026-04-30 16:30:41 | M4-powered iPad Air              | The latest iPad Air model, powered by the M4 chip.
Apple Inc.                   | AAPL       | 2026-04-30 16:30:41 | MacBook Neo                      | A new MacBook model, captivating customers worldwide.
Apple Inc.                   | AAPL       | 2025-10-30 16:30:35 | iPhone 17                        | Apple's latest smartpho

**Note on `llm_seconds`:** this notebook run reuses LLM answers saved in `cache/llm_cache.json`, so `llm_seconds` and `llm_calls_uncached` show 0 here.

In [12]:
print(json.dumps(json.load(open(ex.STATS_JSON)), indent=2))
df.groupby("stock_name").size().sort_values(ascending=False)

{
  "model": "llama3.1:8b",
  "companies_processed": 18,
  "companies_with_products": [
    "AAPL",
    "ADBE",
    "AMD",
    "AMZN",
    "CRM",
    "CSCO",
    "GOOGL",
    "IBM",
    "INTC",
    "META",
    "MSFT",
    "MU",
    "NOW",
    "NVDA",
    "ORCL"
  ],
  "filings_listed": 354,
  "filings_downloaded": 212,
  "filings_sent_to_llm": 98,
  "filings_with_products": 48,
  "products_rejected_validation": 144,
  "llm_seconds": 0.0,
  "llm_calls_uncached": 0,
  "started": "2026-11-04T18:19:36",
  "rows_written": 110,
  "finished": "2026-11-04T18:20:27"
}


stock_name
AMZN     25
NVDA     19
NOW      13
AMD      11
INTC     11
AAPL      9
CRM       6
IBM       4
ORCL      3
GOOGL     2
META      2
MSFT      2
ADBE      1
CSCO      1
MU        1
dtype: int64